# 12F-04 — WLTP/JRC ingest and grain

**Question:** What does this anonymized technical source represent?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
JRC_FILE = REPO_ROOT / "etl/data/raw/wltp_jrc/Data_PV_fleet_2021_EU_PYCSIS.xlsx"
jrc = pd.read_excel(JRC_FILE, sheet_name="Sheet1", engine="openpyxl")
print(f"{jrc.shape[0]:,} rows × {jrc.shape[1]} columns")


## Source-scoped identity and powertrain flags

In [ ]:
jrc[["OEM anon", "Model anon", "Input type", "Fuel type", "is_plugin", "is_hybrid", "is_electric"]].head(10)


In [ ]:
flag_columns = ["is_plugin", "is_hybrid", "is_electric"]
assert all(pd.api.types.is_bool_dtype(jrc[column]) for column in flag_columns)
jrc[flag_columns].fillna(False).astype(bool).value_counts()


## Engineering-field coverage

In [ ]:
engineering_fields = [
    "Vehicle mass (WLTP) [kg]", "wltp|f0 [N]", "wltp|f1 [N/(km/h)]", "wltp|f2 [N/(km/h)2]",
    "Gear box type", "N gears", "Tyre code", "Declared average CO2 emissions value (OEM) [g/km]",
    "Declared electric consumption value (OEM) [Wh/km]", "Electric range (OEM) [km]",
]
coverage = pd.DataFrame({"non_null": jrc[engineering_fields].notna().sum(), "coverage_pct": jrc[engineering_fields].notna().mean() * 100})
coverage.sort_values("coverage_pct", ascending=False)


## First grain check

In [ ]:
grain = pd.Series({
    "source_rows": len(jrc),
    "anonymous_oem_model": len(jrc[["OEM anon", "Model anon"]].drop_duplicates()),
    "oem_model_input_type": len(jrc[["OEM anon", "Model anon", "Input type"]].drop_duplicates()),
    "pycsis_runs": jrc["pycsis_run"].nunique(dropna=True),
})
grain


## What did we learn?

Identity is deliberately anonymized and source-scoped. The source is rich in mass, roadload, gearbox, tire, and declared/simulated result context, but a row should not be promoted to a public commercial identity.